# 데이터 조인

## 조인 목표
1. Profile과 Transcript 데이터 조인 (고객 정보 + 거래/이벤트 정보)
2. Transcript와 Portfolio 데이터 조인 (거래/이벤트 정보 + 프로모션 정보)
3. 최종 통합 데이터셋 생성
4. 코호트 분석 및 퍼널 분석을 위한 데이터 준비


## 1. 라이브러리 및 데이터 불러오기


In [1]:
# 라이브러리 불러오기
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns

# 데이터 경로: ../데이터셋 우선, 없으면 ../data / ../data/전처리_완료_데이터셋
DATA_DIR = "../데이터셋" if os.path.isdir("../데이터셋") else "../data"
CLEAN_DIR = "../데이터셋" if os.path.isdir("../데이터셋") else "../data/전처리_완료_데이터셋"
print("라이브러리 불러오기 완료")


라이브러리 불러오기 완료


In [2]:
# 전처리된 데이터 불러오기
portfolio = pd.read_csv(os.path.join(CLEAN_DIR, "portfolio_clean.csv"), encoding='utf-8-sig')
profile = pd.read_csv(os.path.join(DATA_DIR, "profile.csv"), encoding='utf-8-sig')
transcript = pd.read_csv(os.path.join(CLEAN_DIR, "transcript_clean.csv"), encoding='utf-8-sig')

print("데이터 불러오기 완료")
print(f"Portfolio: {portfolio.shape}")
print(f"Profile: {profile.shape}")
print(f"Transcript: {transcript.shape}")


데이터 불러오기 완료
Portfolio: (10, 12)
Profile: (17000, 6)
Transcript: (306066, 7)


In [3]:
# 데이터 구조 확인
print("=== Portfolio 컬럼 ===")
print(portfolio.columns.tolist())
print(portfolio.head())

print("\n=== Profile 컬럼 ===")
print(profile.columns.tolist())
print(profile.head())

print("\n=== Transcript 컬럼 ===")
print(transcript.columns.tolist())
print(transcript.head())


=== Portfolio 컬럼 ===
['Unnamed: 0', 'reward', 'channels', 'difficulty', 'duration', 'offer_type', 'offer_id', 'channels_parsed', 'channel_email', 'channel_mobile', 'channel_social', 'channel_web']
   Unnamed: 0  reward                              channels  difficulty  \
0           0      10         ['email', 'mobile', 'social']          10   
1           1      10  ['web', 'email', 'mobile', 'social']          10   
2           2       0            ['web', 'email', 'mobile']           0   
3           3       5            ['web', 'email', 'mobile']           5   
4           4       5                      ['web', 'email']          20   

   duration     offer_type                          offer_id  \
0         7           bogo  ae264e3637204a6fb9bb56bc8210ddfd   
1         5           bogo  4d5c57ea9a6940dd891ad53e9dbe8da0   
2         4  informational  3f207df678b143eea3cee63160fa8bed   
3         7           bogo  9b98b8c7a33c4b65b9aebfe6a799e6d9   
4        10       discount  0b1e

## 2. 컬럼명 통일 및 조인 준비


In [4]:
# Portfolio: id 컬럼을 offer_id로 변경 (이미 clean 파일에는 id로 되어있을 수 있음)
if 'id' in portfolio.columns and 'offer_id' not in portfolio.columns:
    portfolio = portfolio.rename(columns={'id': 'offer_id'})

# Profile: id 컬럼을 customer_id로 변경
if 'id' in profile.columns and 'customer_id' not in profile.columns:
    profile = profile.rename(columns={'id': 'customer_id'})

# Transcript: person 컬럼을 customer_id로 변경 (이미 clean 파일에는 변경되어 있을 수 있음)
if 'person' in transcript.columns and 'customer_id' not in transcript.columns:
    transcript = transcript.rename(columns={'person': 'customer_id'})

# Transcript에 offer_id 컬럼이 없으면 value 컬럼에서 파싱 필요
if 'offer_id' not in transcript.columns:
    import ast
    def parse_value(value_str):
        try:
            value_dict = ast.literal_eval(value_str) if isinstance(value_str, str) else value_str
            offer_id = value_dict.get('offer id') or value_dict.get('offer_id')
            amount = value_dict.get('amount')
            return pd.Series({'offer_id': offer_id, 'amount': amount})
        except:
            return pd.Series({'offer_id': None, 'amount': None})
    
    if 'value' in transcript.columns:
        value_parsed = transcript['value'].apply(parse_value)
        transcript['offer_id'] = value_parsed['offer_id']
        transcript['amount'] = value_parsed['amount']

print("컬럼명 통일 완료")
print(f"\nPortfolio 컬럼: {portfolio.columns.tolist()}")
print(f"Profile 컬럼: {profile.columns.tolist()}")
print(f"Transcript 컬럼: {transcript.columns.tolist()}")


컬럼명 통일 완료

Portfolio 컬럼: ['Unnamed: 0', 'reward', 'channels', 'difficulty', 'duration', 'offer_type', 'offer_id', 'channels_parsed', 'channel_email', 'channel_mobile', 'channel_social', 'channel_web']
Profile 컬럼: ['Unnamed: 0', 'gender', 'age', 'customer_id', 'became_member_on', 'income']
Transcript 컬럼: ['Unnamed: 0', 'customer_id', 'event', 'value', 'time', 'offer_id', 'amount']


## 3. 데이터 조인 수행


In [5]:
# 1단계: Profile과 Transcript 조인
# profile[id] = transcript[person] (또는 customer_id)
print("=== 1단계: Profile과 Transcript 조인 ===")
print(f"조인 전 Profile 행 수: {len(profile)}")
print(f"조인 전 Transcript 행 수: {len(transcript)}")

# Profile의 customer_id와 Transcript의 customer_id로 조인
df_profile_transcript = transcript.merge(
    profile,
    on='customer_id',
    how='left',
    suffixes=('', '_profile')
)

print(f"조인 후 행 수: {len(df_profile_transcript)}")
print(f"조인 성공률: {df_profile_transcript['customer_id'].notna().sum() / len(transcript) * 100:.2f}%")
print(f"Profile 정보가 없는 Transcript 행 수: {df_profile_transcript['customer_id'].isna().sum()}")


=== 1단계: Profile과 Transcript 조인 ===
조인 전 Profile 행 수: 17000
조인 전 Transcript 행 수: 306066
조인 후 행 수: 306066
조인 성공률: 100.00%
Profile 정보가 없는 Transcript 행 수: 0


In [6]:
# 2단계: Transcript와 Portfolio 조인
# transcript[offer_id] = portfolio[id] (또는 offer_id)
print("\n=== 2단계: Transcript와 Portfolio 조인 ===")
print(f"조인 전 Portfolio 행 수: {len(portfolio)}")
print(f"조인 전 df_profile_transcript 행 수: {len(df_profile_transcript)}")

# offer_id가 있는 행만 조인 (transaction 이벤트는 offer_id가 없을 수 있음)
df_final = df_profile_transcript.merge(
    portfolio,
    on='offer_id',
    how='left',
    suffixes=('', '_portfolio')
)

print(f"조인 후 행 수: {len(df_final)}")
print(f"Portfolio 정보가 있는 행 수: {df_final['offer_id'].notna().sum()}")
print(f"Portfolio 정보가 없는 행 수 (transaction 등): {df_final['offer_id'].isna().sum()}")



=== 2단계: Transcript와 Portfolio 조인 ===
조인 전 Portfolio 행 수: 10
조인 전 df_profile_transcript 행 수: 306066
조인 후 행 수: 306066
Portfolio 정보가 있는 행 수: 167581
Portfolio 정보가 없는 행 수 (transaction 등): 138485


## 4. 조인 결과 확인


In [7]:
# 최종 데이터셋 정보
print("=== 최종 통합 데이터셋 정보 ===")
print(f"Shape: {df_final.shape}")
print(f"\n컬럼 목록 ({len(df_final.columns)}개):")
print(df_final.columns.tolist())

print("\n=== 데이터 샘플 ===")
print(df_final.head(10))


=== 최종 통합 데이터셋 정보 ===
Shape: (306066, 23)

컬럼 목록 (23개):
['Unnamed: 0', 'customer_id', 'event', 'value', 'time', 'offer_id', 'amount', 'Unnamed: 0_profile', 'gender', 'age', 'became_member_on', 'income', 'Unnamed: 0_portfolio', 'reward', 'channels', 'difficulty', 'duration', 'offer_type', 'channels_parsed', 'channel_email', 'channel_mobile', 'channel_social', 'channel_web']

=== 데이터 샘플 ===
   Unnamed: 0                       customer_id           event  \
0           0  78afa995795e4d85b5d9ceeca43f5fef  offer received   
1           1  a03223e636434f42ac4c3df47e8bac43  offer received   
2           2  e2127556f4f64592b11af22de27a7932  offer received   
3           3  8ec6ce2a7e7949b1bf142def7d0e0586  offer received   
4           4  68617ca6246f4fbc85e91a2a49552598  offer received   
5           5  389bc3fa690240e798340f5a15918d5c  offer received   
6           6  c4863c7985cf408faee930f111475da3  offer received   
7           7  2eeac8d8feae4a8cad5a6af0499a211d  offer received   
8    

In [8]:
# 조인 결과 통계
print("=== 조인 결과 통계 ===")
print(f"\n1. 고객 정보 조인:")
print(f"   - Profile 정보가 있는 행: {df_final['customer_id'].notna().sum():,}건")
print(f"   - Profile 정보가 없는 행: {df_final['customer_id'].isna().sum():,}건")

print(f"\n2. 프로모션 정보 조인:")
print(f"   - Portfolio 정보가 있는 행: {df_final['offer_id'].notna().sum():,}건")
print(f"   - Portfolio 정보가 없는 행: {df_final['offer_id'].isna().sum():,}건")

print(f"\n3. 이벤트 타입별 분포:")
print(df_final['event'].value_counts())

print(f"\n4. 프로모션 타입별 분포 (offer_id가 있는 경우):")
if 'offer_type' in df_final.columns:
    print(df_final[df_final['offer_id'].notna()]['offer_type'].value_counts())


=== 조인 결과 통계 ===

1. 고객 정보 조인:
   - Profile 정보가 있는 행: 306,066건
   - Profile 정보가 없는 행: 0건

2. 프로모션 정보 조인:
   - Portfolio 정보가 있는 행: 167,581건
   - Portfolio 정보가 없는 행: 138,485건

3. 이벤트 타입별 분포:
event
transaction        138485
offer received      76277
offer viewed        57725
offer completed     33579
Name: count, dtype: int64

4. 프로모션 타입별 분포 (offer_id가 있는 경우):
offer_type
bogo             71617
discount         69898
informational    26066
Name: count, dtype: int64


## 5. 코호트 분석을 위한 데이터 준비


In [9]:
# became_member_on 컬럼을 날짜 형식으로 변환
if 'became_member_on' in df_final.columns:
    # became_member_on이 숫자 형식 (예: 20170212)인 경우 날짜로 변환
    df_final['became_member_on'] = pd.to_datetime(
        df_final['became_member_on'].astype(str), 
        format='%Y%m%d', 
        errors='coerce'
    )
    
    # 회원가입 연도/월 추출
    df_final['member_year'] = df_final['became_member_on'].dt.year
    df_final['member_month'] = df_final['became_member_on'].dt.month
    df_final['member_year_month'] = df_final['became_member_on'].dt.to_period('M')
    
    print("=== 회원가입 시점 정보 ===")
    print(f"회원가입 연도별 분포:")
    print(df_final.groupby('member_year')['customer_id'].nunique().sort_index())
    print(f"\n회원가입 월별 분포:")
    print(df_final.groupby('member_month')['customer_id'].nunique().sort_index())


=== 회원가입 시점 정보 ===
회원가입 연도별 분포:
member_year
2013     286
2014     691
2015    1830
2016    3526
2017    6469
2018    4198
Name: customer_id, dtype: int64

회원가입 월별 분포:
member_month
1     1525
2     1202
3     1329
4     1315
5     1307
6     1265
7     1359
8     1610
9     1515
10    1568
11    1449
12    1556
Name: customer_id, dtype: int64


In [10]:
# time 컬럼을 날짜로 변환 (시작 시점 기준으로 변환)
# time은 시간 단위로 되어있을 것으로 가정 (예: 0 = 시작 시점, 168 = 1주일 후)
# 실제 시작 날짜가 필요하면 별도로 설정 필요

print("=== Time 정보 확인 ===")
print(f"Time 범위: {df_final['time'].min()} ~ {df_final['time'].max()}")
print(f"Time 통계:")
print(df_final['time'].describe())


=== Time 정보 확인 ===
Time 범위: 0 ~ 714
Time 통계:
count    306066.000000
mean        366.329517
std         200.316996
min           0.000000
25%         186.000000
50%         408.000000
75%         528.000000
max         714.000000
Name: time, dtype: float64


## 6. 퍼널 분석을 위한 데이터 준비


In [11]:
# 이벤트 타입별 확인
print("=== 이벤트 타입별 분포 ===")
event_counts = df_final['event'].value_counts()
print(event_counts)
print(f"\n총 이벤트 수: {len(df_final):,}건")

# 프로모션 관련 이벤트만 필터링
promotion_events = ['offer received', 'offer viewed', 'offer completed']
df_promotion = df_final[df_final['event'].isin(promotion_events)].copy()

print(f"\n=== 프로모션 관련 이벤트 ===")
print(f"프로모션 관련 이벤트 수: {len(df_promotion):,}건")
print(df_promotion['event'].value_counts())


=== 이벤트 타입별 분포 ===
event
transaction        138485
offer received      76277
offer viewed        57725
offer completed     33579
Name: count, dtype: int64

총 이벤트 수: 306,066건

=== 프로모션 관련 이벤트 ===
프로모션 관련 이벤트 수: 167,581건
event
offer received     76277
offer viewed       57725
offer completed    33579
Name: count, dtype: int64


In [12]:
# 고객별 프로모션 퍼널 확인 (샘플)
print("=== 고객별 프로모션 퍼널 샘플 ===")
sample_customers = df_promotion['customer_id'].unique()[:5]

for customer_id in sample_customers:
    customer_events = df_promotion[df_promotion['customer_id'] == customer_id].sort_values('time')
    print(f"\n고객 ID: {customer_id}")
    print(customer_events[['event', 'offer_id', 'offer_type', 'time']].head(10))


=== 고객별 프로모션 퍼널 샘플 ===

고객 ID: 78afa995795e4d85b5d9ceeca43f5fef
                  event                          offer_id     offer_type  time
0        offer received  9b98b8c7a33c4b65b9aebfe6a799e6d9           bogo     0
15560      offer viewed  9b98b8c7a33c4b65b9aebfe6a799e6d9           bogo     6
47523   offer completed  9b98b8c7a33c4b65b9aebfe6a799e6d9           bogo   132
53096    offer received  5a8bc65990b245e5a138643cd4eb9837  informational   168
85184      offer viewed  5a8bc65990b245e5a138643cd4eb9837  informational   216
150390   offer received  ae264e3637204a6fb9bb56bc8210ddfd           bogo   408
163167     offer viewed  ae264e3637204a6fb9bb56bc8210ddfd           bogo   408
201286   offer received  f19421c1d4aa40978ebb69ca19b0e20d           bogo   504
218107  offer completed  ae264e3637204a6fb9bb56bc8210ddfd           bogo   510
218108  offer completed  f19421c1d4aa40978ebb69ca19b0e20d           bogo   510

고객 ID: a03223e636434f42ac4c3df47e8bac43
                 event    

## 7. 최종 데이터셋 저장


In [13]:
# 최종 통합 데이터셋 저장
output_path = os.path.join(CLEAN_DIR, "starbucks_merged.csv")
df_final.to_csv(output_path, index=False, encoding='utf-8-sig')

print(f"최종 통합 데이터셋 저장 완료: {output_path}")
print(f"저장된 데이터 Shape: {df_final.shape}")
print(f"\n저장된 컬럼 목록:")
print(df_final.columns.tolist())


최종 통합 데이터셋 저장 완료: ../data/전처리_완료_데이터셋\starbucks_merged.csv
저장된 데이터 Shape: (306066, 26)

저장된 컬럼 목록:
['Unnamed: 0', 'customer_id', 'event', 'value', 'time', 'offer_id', 'amount', 'Unnamed: 0_profile', 'gender', 'age', 'became_member_on', 'income', 'Unnamed: 0_portfolio', 'reward', 'channels', 'difficulty', 'duration', 'offer_type', 'channels_parsed', 'channel_email', 'channel_mobile', 'channel_social', 'channel_web', 'member_year', 'member_month', 'member_year_month']


## 8. 다음 단계 안내

조인된 데이터를 기반으로 다음 분석을 진행할 수 있습니다:

1. **코호트 분석**: 회원가입 시점(became_member_on)을 기준으로 고객 코호트 정의
2. **퍼널 분석**: 프로모션 수신 → 반응(조회) → 전환(완료) 단계별 전환율 분석
3. **코호트 × 프로모션 매칭**: 코호트별로 적합한 프로모션 유형 도출
